# Unit 6 hands-on: fine-tune SpeechT5 for text-to-speech

This notebook walks you through the Hugging Face Audio Course **Unit 6 hands-on** from start to
finish.

**What you'll do:** take `microsoft/speecht5_tts` and *fine-tune* it on a set of recorded English
dialect speech, so it learns to speak in those voices. Then you'll push the model to the Hub.

**How you're graded:** you aren't, on a number. The course says this exercise *"will focus on
practicing the skills rather than achieving a certain metric value."* You pass as soon as a model
tagged **`text-to-speech`** exists under your account. That makes this the most forgiving of the four
hands-ons: there is no threshold to chase, and no way to "just miss" it.

**What that means in practice:** the only things that can go wrong are mechanical. This notebook is
built to head each of them off, and every one is called out where it happens.

> **TTS is two models, not one.** SpeechT5 turns text into a **log-mel spectrogram**; a separate
> **vocoder** (`microsoft/speecht5_hifigan`) turns that spectrogram into audio you can hear. You only
> fine-tune the first one. The vocoder stays frozen and is only needed when you want to listen.

## Step 1 — Install the libraries

`sentencepiece` is **not optional here.** SpeechT5's tokenizer is sentencepiece-only — there is no
fast variant — so without it `SpeechT5Processor.from_pretrained(...)` raises `ImportError` rather than
falling back to something slower.

In [ ]:
!pip install -q "transformers>=4.46,<5" "datasets==3.6.0" sentencepiece \
                "accelerate>=0.30" "soundfile>=0.12.1" "librosa>=0.10" tensorboard

> If a later cell raises a strange import error, do **Runtime → Restart session**, then run again
> from Step 2 (you don't need to re-run Step 1 — the packages are already installed).

## Step 2 — Check the GPU and authenticate with Hugging Face

The first lines confirm a GPU is active. Then the notebook needs a token so it can upload your model
at the end.

**Use a Colab Secret, not a login prompt.** Click the **key icon** in the left sidebar, add a secret
named exactly `HF_TOKEN`, paste a **write** token as its value, and turn **Notebook access** on.
`huggingface_hub` picks it up automatically — there is nothing to call.

Create the token at **https://huggingface.co/settings/tokens → Create new token → type "Write"**.

> **The Colab vault beats everything else.** `get_token()` resolves in the order *Colab secret →
> `HF_TOKEN` environment variable → token file*, and `notebook_login()` writes to the **token file**,
> which is last. So a stale or invalid `HF_TOKEN` secret silently overrides a login that looked like
> it succeeded. After changing the secret's value you must **restart the runtime** — the vault is read
> once per session and cached.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(),
      "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set Runtime -> T4 GPU!")

Now confirm the token is found and actually works. `whoami()` raises if it is invalid, so this cell is
your fast failure: a three-second error here beats a `401` when the trainer tries to create your repo
several steps later (which then shows up as a confusing `NameError`).

In [ ]:
from huggingface_hub import get_token, whoami

assert get_token(), (
    "No token found. Add a secret named HF_TOKEN in Colab (key icon, left sidebar), "
    "paste a WRITE token as its value, and enable Notebook access."
)

info = whoami()   # raises if the token is expired, revoked or mistyped
print("logged in as:", info["name"])
print("token role  :", info.get("auth", {}).get("accessToken", {}).get("role", "(fine-grained)"))

## Step 3 — Load the dataset

The hands-on says *"fine-tune the SpeechT5 model on a dataset of your choosing"*, so we use
**`ylacombe/english_dialects`**, config `northern_female`: **750 clips from 5 speakers** (150 each) of
read English in a northern accent, about 395 MB.

Why not the course's VoxPopuli Dutch? That config is **10.4 GB** for the train split. Since there is
no metric to hit, the download would buy nothing but risk. This dataset is also parquet-native, so no
`trust_remote_code` is involved.

SpeechT5 expects **16 kHz** audio, so we resample on load.

In [ ]:
from datasets import load_dataset, Audio

DATASET_ID = "ylacombe/english_dialects"
CONFIG     = "northern_female"

ds = load_dataset(DATASET_ID, CONFIG, split="train")
ds = ds.cast_column("audio", Audio(sampling_rate=16_000))

print(ds)
print("speakers:", sorted(set(ds["speaker_id"])))
print("example :", repr(ds[0]["text"]))

## Step 4 — The processor, and the vocabulary trap

`SpeechT5Processor` bundles two things: a **tokenizer** for the text and a **feature extractor** that
turns target audio into the log-mel spectrogram the model learns to predict.

**The trap:** SpeechT5's tokenizer vocabulary is only **81 tokens** — character-level, and covering
almost no punctuation. Anything outside it becomes `<unk>` **silently**. No warning, no error; the
model just learns from mangled text.

> **Audit by tokenizing, not by comparing against `get_vocab()` keys.** SpeechT5 is a *sentencepiece*
> model, so a space is stored as the word-boundary marker `▁` (U+2581). A literal `" "` is therefore
> absent from `get_vocab()` even though it tokenizes perfectly — and a naive
> `set(text) - set(get_vocab())` check rejects every row that contains a space, which is every row.
> Asking the tokenizer whether it emits `<unk>` is the only reliable test.

In [ ]:
from transformers import SpeechT5Processor

MODEL_ID = "microsoft/speecht5_tts"
processor = SpeechT5Processor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
unk_id = tokenizer.unk_token_id

# len(get_vocab()) is 81 and matches config.vocab_size, the model's embedding table.
# tokenizer.vocab_size reports 79: it excludes the added <mask> and <ctc_blank>.
print("vocab size:", len(tokenizer.get_vocab()), "| <unk> id:", unk_id)

# Ask the tokenizer what it cannot represent, rather than guessing from
# get_vocab() keys. This is a sentencepiece model: a space is stored as the
# word-boundary marker U+2581, so a literal " " looks absent from those keys
# even though it tokenizes perfectly. Comparing characters to vocab keys
# therefore rejects every row that contains a space, which is all of them.
def unsupported_chars(texts):
    chars = set("".join(texts))
    return {c for c in chars
            if unk_id in tokenizer(c, add_special_tokens=False).input_ids}

print("characters in the dataset :", "".join(sorted(set("".join(ds["text"])))))
print("tokenizer cannot represent:", "".join(sorted(unsupported_chars(ds["text"]))))

Now clean the text so nothing falls outside the vocabulary. Lowercasing does most of the work and
`REPLACEMENTS` maps the accented characters and typographic punctuation onto ASCII.

**Digits are the awkward case.** `0123589` are genuinely absent from an 81-symbol character
vocabulary, and turning "1984" into speakable words reliably is its own project. So after replacing
what can be replaced, we **drop** the remaining rows.

Note the cell asserts **both** that something survived *and* that what survived is clean. The second
check alone is vacuously true on an empty dataset — which is precisely how an earlier version of this
notebook managed to discard all 750 rows while printing a success message.

In [ ]:
REPLACEMENTS = [
    ("\u2019", "'"),        # right single quote -> ASCII apostrophe
    ("\u2018", "'"),
    ("\u201c", ""),         # curly double quotes -> dropped
    ("\u201d", ""),
    ("\u2014", " "),        # em dash -> space
    ("\u2013", " "),        # en dash -> space
    ("\u00e1", "a"),        # a-acute
    ("\u00f4", "o"),        # o-circumflex
    ("\u00fc", "u"),        # u-umlaut
    ("\u00a3", " pounds "),  # sterling sign
    ("-",      " "),
    (";",      ","),
    (":",      ","),
    ('"',      ""),
]

def clean_text(example):
    text = example["text"].lower()
    for old, new in REPLACEMENTS:
        text = text.replace(old, new)
    example["text"] = " ".join(text.split())   # collapse whitespace
    return example

ds = ds.map(clean_text)

# Digits have no place in an 81-symbol character vocabulary, and spelling them
# out reliably is its own project. Drop the rows that still tokenize to <unk>.
def is_representable(text):
    return unk_id not in tokenizer(text, add_special_tokens=False).input_ids

before = len(ds)
ds = ds.filter(is_representable, input_columns=["text"])
print("dropped %d of %d rows the tokenizer cannot represent" % (before - len(ds), before))

# Both assertions matter. The second one alone would pass vacuously on an empty
# dataset, which is exactly how an earlier version of this cell hid the fact
# that it had discarded everything.
assert len(ds) > 0, "the filter removed every row - re-read the Step 4 audit before continuing"
assert not unsupported_chars(ds["text"]), "still unrepresentable after cleaning"

print("kept %d rows, all representable" % len(ds))
print(repr(ds[0]["text"]))

## Step 5 — Speaker embeddings

SpeechT5 is a **multi-speaker** model: alongside the text it takes a 512-dimensional **x-vector**
describing *whose voice* to use. Without it the model has no way to know which of our 5 speakers a
given clip belongs to, and training collapses toward an average mush.

The course computes these with SpeechBrain's `spkrec-xvect-voxceleb`. **We don't**, for two reasons:
SpeechBrain 1.x renamed `speechbrain.pretrained` to `speechbrain.inference` (so the course's import
line is a straight `ImportError` on any current install), and it drags in `torchaudio`.

Instead we use the precomputed **`Matthijs/cmu-arctic-xvectors`** and assign each of our 5 speakers
its own fixed vector. The x-vector then acts as a consistent *speaker code* rather than a true timbre
embedding — the model still learns the voices, and since this unit has no metric, nothing is lost.

> Note the x-vectors live in the **`validation`** split, not `train`.

In [ ]:
import torch
from datasets import load_dataset as _load

xvectors = _load("Matthijs/cmu-arctic-xvectors", split="validation")
filenames = xvectors["filename"]

# one distinct CMU ARCTIC voice per dataset speaker
VOICES = ["slt", "clb", "bdl", "rms", "ksp"]
speakers = sorted(set(ds["speaker_id"]))
assert len(speakers) <= len(VOICES), "more speakers than voices - extend VOICES"

def first_index_for(voice):
    prefix = "cmu_us_%s_" % voice
    return next(i for i, f in enumerate(filenames) if f.startswith(prefix))

SPEAKER_VEC = {
    sid: torch.tensor(xvectors[first_index_for(v)]["xvector"])
    for sid, v in zip(speakers, VOICES)
}

for sid, v in zip(speakers, VOICES):
    print("speaker %-6s -> cmu_us_%s  (dim %d)" % (sid, v, len(SPEAKER_VEC[sid])))

## Step 6 — Turn text and audio into model inputs

One call does both halves: `text=` produces `input_ids`, and `audio_target=` produces `labels`, which
for TTS are a **log-mel spectrogram** (80 bins per frame) rather than token ids.

> **`example["labels"] = example["labels"][0]` is required here.** The processor returns the
> spectrogram wrapped in a batch dimension of one, and the collator expects it unwrapped. This is the
> exact *opposite* of Unit 5, where indexing `[0]` was the bug — there the labels are a flat list of
> token ids and `[0]` would take a single integer. Same-looking line, opposite meaning.

We also drop clips whose tokenized text exceeds 200 tokens; long sequences blow up decoder memory and
destabilise training.

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]
    out = processor(
        text=example["text"],
        audio_target=audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_attention_mask=False,
    )
    out["labels"] = out["labels"][0]                      # <- required (see note above)
    out["speaker_embeddings"] = SPEAKER_VEC[example["speaker_id"]]
    return out

processed = ds.map(prepare_dataset, remove_columns=ds.column_names, num_proc=1)
processed = processed.filter(lambda ids: len(ids) <= 200, input_columns=["input_ids"])

processed = processed.train_test_split(test_size=0.1, seed=42)
print(processed)
print("input_ids len :", len(processed["train"][0]["input_ids"]))
print("labels shape  :", len(processed["train"][0]["labels"]), "frames x",
      len(processed["train"][0]["labels"][0]), "mel bins")

## Step 7 — The data collator

This is the fiddliest part of the whole unit, and it is where a hand-written version usually fails
with a shape error deep inside the loss. Three things have to happen:

1. **Pad** text and spectrograms to the longest item in the batch.
2. **Mask padding with `-100`** so the loss ignores it.
3. **Truncate target lengths to a multiple of `reduction_factor`** (which is `2` for this checkpoint).
   SpeechT5's decoder emits 2 spectrogram frames per step, so a target length that isn't even leaves
   the loss misaligned against the predictions.

We also delete `decoder_attention_mask` — the model does not take it during training.

> **`torch.tensor(speaker_feats)`, not `torch.stack(...)`.** The x-vectors were written into the
> dataset in Step 6, and anything stored in a 🤗 Dataset round-trips through Arrow, so they come back
> as plain Python **lists** rather than tensors. `torch.stack` needs tensors and raises
> `TypeError: expected Tensor as element 0 in argument 0, but got list`.

The last two lines build a real batch and print its shapes. Run it: a collator bug caught here costs
you seconds, whereas the same bug found inside `trainer.train()` costs a GPU session.

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class TTSDataCollatorWithPadding:
    processor: Any
    reduction_factor: int = 2

    def __call__(self, features):
        input_ids      = [{"input_ids": f["input_ids"]} for f in features]
        label_features = [{"input_values": f["labels"]} for f in features]
        speaker_feats  = [f["speaker_embeddings"] for f in features]

        batch = self.processor.pad(
            input_ids=input_ids, labels=label_features, return_tensors="pt"
        )

        # ignore padded frames in the loss
        batch["labels"] = batch["labels"].masked_fill(
            batch.decoder_attention_mask.unsqueeze(-1).ne(1), -100
        )
        del batch["decoder_attention_mask"]     # not used during training

        # round target lengths down to a multiple of the reduction factor
        target_lengths = torch.tensor([len(f["input_values"]) for f in label_features])
        target_lengths = target_lengths.new(
            [l - l % self.reduction_factor for l in target_lengths]
        )
        batch["labels"] = batch["labels"][:, : max(target_lengths)]

        batch["speaker_embeddings"] = torch.tensor(speaker_feats)   # tensor(), not stack(): see note
        return batch

collator = TTSDataCollatorWithPadding(processor=processor)

# sanity check on a real batch before training touches it
_b = collator([processed["train"][i] for i in range(2)])
print({k: tuple(v.shape) for k, v in _b.items()})

## Step 8 — Load the model

`use_cache=False` because gradient checkpointing (Step 9) recomputes activations instead of storing
them, and the two are incompatible.

In [ ]:
from transformers import SpeechT5ForTextToSpeech

model = SpeechT5ForTextToSpeech.from_pretrained(MODEL_ID)
model.config.use_cache = False

print("reduction_factor     :", model.config.reduction_factor)
print("speaker_embedding_dim:", model.config.speaker_embedding_dim)

## Step 9 — Training configuration

Four settings here are not style choices — leave any of them out and the run breaks, usually quietly:

- **`label_names=["labels"]`** — without it the `Trainer` cannot find the labels inside SpeechT5's
  output object, and your eval loss is silently never computed.
- **`remove_unused_columns=False`** — without it `speaker_embeddings` is pruned from the dataset
  before the collator ever sees it.
- **`predict_with_generate` must stay off.** SpeechT5's `generate()` returns a *spectrogram*, not
  token ids, so the generate-during-eval path breaks on it. (`can_generate()` returns `True` only so
  the `GenerationConfig` plumbing works — it is not a normal generative model.)
- **`gradient_checkpointing_kwargs={"use_reentrant": False}`** — left unset, PyTorch falls back to its
  legacy reentrant checkpointing, and the first backward pass dies with `Trying to backward through
  the graph a second time`.

**`max_steps=1000`** takes roughly 30–40 minutes on a T4. The course suggests 4000, which is about
four hours and would eat your whole daily Colab quota for no benefit — there is no metric to chase, so
the requirement is simply that the loss goes down and a model exists.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

REPO_NAME = "speecht5_finetuned_english_dialects"

training_args = Seq2SeqTrainingArguments(
    output_dir=REPO_NAME,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,          # effective batch 32
    per_device_eval_batch_size=8,
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=1000,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},   # NOT optional
    fp16=True,
    eval_strategy="steps",                  # NOT evaluation_strategy (removed)
    save_strategy="steps",
    eval_steps=250,
    save_steps=250,
    save_total_limit=2,
    logging_steps=25,
    report_to=["tensorboard"],
    label_names=["labels"],                 # NOT optional
    remove_unused_columns=False,            # NOT optional
    push_to_hub=True,
    hub_strategy="end",                     # one upload at the end
    # predict_with_generate intentionally absent - generate() returns a spectrogram
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed["train"],
    eval_dataset=processed["test"],
    data_collator=collator,
    processing_class=processor,             # current API (older tutorials used tokenizer=)
)

> **If this cell raises `401` or `403` on `/api/repos/create`**, fix your token before going on: the
> repo is created here, the moment `push_to_hub=True` is set, before any training, so you lose no
> time. `401` means the `HF_TOKEN` secret is invalid (replace it, then restart the runtime); `403`
> means it is read-only. If this cell fails, `trainer` is never defined and the next cells raise
> `NameError` — fix the token, not the `NameError`.

## Step 10 — Train (this is the ~35 minute step)

Watch `Training Loss` and `Validation Loss` in the table. They should fall steadily. There is no
metric threshold — a decreasing loss is the whole requirement.

Keep the Colab tab open and visible so the runtime doesn't disconnect.

In [ ]:
trainer.train()

## Step 11 — Listen to what you trained

Now the vocoder earns its keep: `generate_speech` gives a spectrogram, and
`microsoft/speecht5_hifigan` turns it into audio.

Pass one of your speaker vectors so the model knows which voice to use.

In [ ]:
from transformers import SpeechT5HifiGan
import IPython.display as ipd

vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

text = "the sun provides energy for life on earth"
inputs = processor(text=text, return_tensors="pt")
speaker = SPEAKER_VEC[sorted(SPEAKER_VEC)[0]].unsqueeze(0)

model.eval()
with torch.no_grad():
    speech = model.to("cpu").generate_speech(inputs["input_ids"], speaker, vocoder=vocoder)

print("generated", len(speech) / 16_000, "seconds")
ipd.Audio(speech.numpy(), rate=16_000)

## Step 12 — Push to the Hub (this is what counts for the certificate)

`tasks="text-to-speech"` is the one kwarg that matters — it is what puts the `text-to-speech` pipeline
tag on your model, and that tag is the entire pass condition for this unit.

Unlike Units 4 and 5, the grader does **not** check the `datasets:` field here, so the dataset choice
is unconstrained.

In [ ]:
kwargs = {
    "dataset_tags": DATASET_ID,
    "dataset": "English Dialects (%s)" % CONFIG,
    "model_name": REPO_NAME,
    "finetuned_from": MODEL_ID,
    "tasks": "text-to-speech",      # <- the pass condition
}
trainer.push_to_hub(**kwargs)

## Step 13 — Confirm the tag (do not skip this)

**This is the single most likely way Unit 6 silently fails.** The grader finds your model by listing
your account filtered on the `text-to-speech` pipeline tag. If the Hub inferred a different tag —
*Text-to-Audio*, *Unconditional Audio Generation*, or nothing at all — your model is invisible to it,
no matter how well it trained.

The cell below checks exactly what the grader checks.

In [ ]:
from huggingface_hub import HfApi

api  = HfApi()
me   = whoami()["name"]
repo = "%s/%s" % (me, REPO_NAME)

info = api.model_info(repo)
print("repo        :", repo)
print("pipeline_tag:", info.pipeline_tag)
print("private     :", info.private)

found = [m.modelId for m in api.list_models(author=me, filter=["text-to-speech"])]
print("\nvisible to the grader:", found)

ok = info.pipeline_tag == "text-to-speech" and not info.private and repo in found
print("\nPASS" if ok else "\nNOT YET - see the cell below")

## Step 14 — Set the pipeline tag

If Step 13 said `NOT YET` with `pipeline_tag: text-to-audio`, that is the **expected** result, not bad
luck. `trainer.push_to_hub(tasks="text-to-speech")` only populates the model-index; it writes no
`pipeline_tag` to the card. With that field absent the Hub infers a tag from the architecture, and for
SpeechT5 it infers `text-to-audio`. The grader queries `text-to-speech`, so your model is invisible
until you set it explicitly.

Setting it is correct, not a workaround: this genuinely *is* a text-to-speech model, and this unit has
no metric, so there is no result being altered — you are correcting an inference the Hub made about
your architecture.

In [ ]:
from huggingface_hub import metadata_update

metadata_update(repo, {"pipeline_tag": "text-to-speech"}, overwrite=True)
print("pipeline_tag written to the model card")

info  = api.model_info(repo)
found = [m.modelId for m in api.list_models(author=me, filter=["text-to-speech"])]
print("pipeline_tag         :", info.pipeline_tag)
print("visible to the grader:", found)

ok = info.pipeline_tag == "text-to-speech" and not info.private and repo in found
print("\nPASS - Unit 6 complete" if ok else
      "\nTag is set but the search index may lag by a minute; re-run this cell.")

> The `pipeline_tag` on the model page updates immediately, but `list_models` reads a **search index**
> that can lag by up to a minute. If the tag is right and `visible to the grader` is still empty, wait
> and re-run the cell rather than changing anything.

You can also do this by hand: open the model's `README.md` on the Hub, click the pencil icon, and add
`pipeline_tag: text-to-speech` to the YAML frontmatter at the top.

## Troubleshooting

- **`ImportError` on `SpeechT5Processor`** — `sentencepiece` didn't install. Re-run Step 1, then
  **Runtime → Restart session**. SpeechT5's tokenizer has no fast variant, so there is no fallback.
- **`401 Unauthorized ... /api/repos/create`**, or `Invalid user token. The token from Google Colab
  vault is invalid` — the token is expired, revoked or mistyped. Update the `HF_TOKEN` Colab secret
  with a fresh **write** token, then **restart the runtime** (the vault is cached per session).
- **`403 Forbidden ... create`** — the token is valid but **read-only**. (401 = bad token,
  403 = wrong scope.)
- **`RuntimeError: Trying to backward through the graph a second time`** — you dropped
  `gradient_checkpointing_kwargs={"use_reentrant": False}` from Step 9. Put it back, or remove
  `gradient_checkpointing=True` entirely.
- **Shape error inside the loss** — almost always the collator. Check that target lengths are
  truncated to a multiple of `reduction_factor` (2) and that `labels` was unwrapped with
  `out["labels"] = out["labels"][0]` in Step 6.
- **`KeyError: 'speaker_embeddings'`** — `remove_unused_columns=False` is missing from Step 9, so the
  column was pruned before the collator ran.
- **`TypeError: expected Tensor as element 0 in argument 0, but got list`** — the collator is using
  `torch.stack` on the speaker embeddings. They come back from the dataset as lists, so it must be
  `torch.tensor`.
- **`AssertionError: the filter removed every row`** in Step 4 — the cleanup discarded everything.
  Read the audit output above it: if a character you expect to be fine (especially a space) is listed
  as unrepresentable, the check is comparing against `get_vocab()` keys instead of tokenizing.
- **`AssertionError: still unrepresentable after cleaning`** in Step 4 — the audit found a character
  `REPLACEMENTS` does not cover. Add a mapping for it. Do not skip the assert: unmapped characters
  become `<unk>` without any warning.
- **`IndexError: Invalid key: 0 is out of bounds for size 0`** — a filter emptied the dataset a cell
  or two earlier. The `IndexError` is the symptom, not the cause.
- **Step 13 reports `pipeline_tag: text-to-audio`** — expected. Run Step 14. `tasks=` in
  `push_to_hub` does not write a `pipeline_tag`, and without one the Hub infers `text-to-audio` from
  the SpeechT5 architecture.
- **Eval loss is never reported** — `label_names=["labels"]` is missing from Step 9.
- **The generated audio is noise or silence** — check Step 11 passes a speaker embedding with a batch
  dimension, `.unsqueeze(0)`, and that the vocoder is on the same device as the model.
- **`CUDA out of memory`** — halve `per_device_train_batch_size` to 8 and double
  `gradient_accumulation_steps` to 4.
- **Runtime disconnected mid-training** — Colab's free tier disconnects when idle. Keep the tab open
  and visible, or lower `max_steps`.